# CIFAR-100 Vision Transformer training

Trains the primary Swin model and the parameter-matched ViT baseline.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine to start).

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

`requirements.txt` has no version pins, so this leaves Colab's
preinstalled, GPU-matched `torch`/`torchvision`/`numpy` untouched instead
of forcing a version that conflicts with other preinstalled packages.

In [ ]:
!git clone https://github.com/dparkar00/Assignment5.git
%cd Assignment5
!pip install -r requirements.txt -q

## 2. (Recommended) Mount Google Drive

Colab runtimes disconnect and wipe local disk after periods of inactivity.
Mounting Drive lets you point checkpoints/logs somewhere that survives a
disconnect, so a long run isn't lost. Skip this cell if you're fine
re-running from scratch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/vision_transformer_runs

If you mounted Drive, point the configs' `checkpoint_dir` there before
training so checkpoints persist. This edits both YAML configs in place.

In [ ]:
import re

drive_base = "/content/drive/MyDrive/vision_transformer_runs"
for path, name in [("configs/primary.yaml", "primary"), ("configs/vit_baseline.yaml", "vit_baseline")]:
    with open(path) as f:
        text = f.read()
    text = re.sub(r"checkpoint_dir:.*", f"checkpoint_dir: {drive_base}/checkpoints/{name}", text)
    with open(path, "w") as f:
        f.write(text)
print("Updated checkpoint_dir in both configs.")

## 3. Log in to Weights & Biases

You'll be prompted to paste your API key (from wandb.ai/authorize).

In [ ]:
import wandb
wandb.login()

## 4. Sanity check: run the full pipeline on synthetic data first

Confirms the training loop, checkpointing, and CSV logging all work on
*this* runtime/GPU before committing to a long real run. No dataset
download, no W&B calls.

In [ ]:
!python -m src.train --config configs/primary.yaml --dry-run --epochs 2

## 5. Train both models: Swin (primary) and ViT (baseline)

**Both models must be trained** -- the assignment requires a primary
Swin/hybrid model AND a parameter-matched plain ViT baseline, both from
random initialization, under identical experimental controls (Part 2).

This cell runs both, back-to-back, in one place, so it's not possible to
accidentally run only one. Each call to `train()` prints
`Pretrained weights: False` at the start -- confirming neither model
loads any pretrained weights, per the assignment's requirement.

In [ ]:
# Each line below launches a separate OS process, so memory from the
# first run is fully released by the OS before the second starts -- safer
# than importing train() and calling it twice in one long-lived process.
!python -m src.train --config configs/primary.yaml
!python -m src.train --config configs/vit_baseline.yaml

### Optional: run just one model

Only needed if you're re-running a single model after a fix (e.g. you
changed a hyperparameter for just the baseline). For a normal full run,
use the combined cell above instead -- these two are here so a partial
re-run doesn't require re-doing the one that already finished.

In [ ]:
!python -m src.train --config configs/primary.yaml

In [ ]:
!python -m src.train --config configs/vit_baseline.yaml

## 6. Verify both models actually finished

Checks for the artifacts both runs are supposed to produce: a CSV log
and a `best.pt` checkpoint for each model. Fails loudly (rather than
silently) if either is missing, so you don't discover it's missing only
once you're writing the report.

In [ ]:
import csv
from pathlib import Path

runs = [
    ("Swin (primary)", "logs/primary_training.csv", "checkpoints/primary/best.pt"),
    ("ViT (baseline)", "logs/vit_training.csv", "checkpoints/vit_baseline/best.pt"),
]

all_ok = True
for name, log_path, checkpoint_path in runs:
    log_ok = Path(log_path).exists()
    ckpt_ok = Path(checkpoint_path).exists()
    all_ok = all_ok and log_ok and ckpt_ok
    status = "OK" if (log_ok and ckpt_ok) else "MISSING"
    print(f"[{status}] {name}: log={log_ok}  checkpoint={ckpt_ok}")
    if log_ok:
        with open(log_path) as f:
            rows = list(csv.DictReader(f))
        if rows:
            last = rows[-1]
            print(
                f"    epochs logged: {len(rows)}  "
                f"final val_accuracy: {float(last['val_accuracy']):.4f}"
            )

if not all_ok:
    raise RuntimeError("At least one model is missing its log or checkpoint -- see above.")
print("\nBoth models trained and verified.")

## 7. Pull the exported logs back down

These CSVs are required in the final submission independent of W&B. If you
mounted Drive, copy them there too so they survive a disconnect.

In [ ]:
!cp logs/*.csv /content/drive/MyDrive/vision_transformer_runs/ 2>/dev/null || echo "Drive not mounted -- logs are in ./logs/ locally only."
!ls -la logs/